# Step 4. Plot and package CTD profiles

Makes publication-style vertical profile plots from the v2 products, per cruise:

* **CTD variables** from the 1 m downcast profiles (`L2/CTD/CTD_1m_down/*.cnv`) — temperature,
  salinity, oxygen, fluorescence, etc. vs depth.
* **Nitrate** from the merged CTD+SUNA product (`L2/SUNA/*_SUNA_1s.csv`) where a cast has SUNA data.

Outputs: per-cast profile figures and per-variable multi-cast overlays, written to
`L2/CTD/profile_plots/`. It also writes a lab-friendly CSV per cast (readable column names) to the
same folder, so profiles can be shared with colleagues without the cryptic Sea-Bird codes.

This is a clean v2 re-implementation of the v1 `step06` plotting notebook, wired to the cruise-centric
layout and validated on real P45_06 data.


## 1. Imports and settings

In [1]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # file output; no interactive backend needed
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)


In [2]:
# ===========================================================================
# Paths (match step01/02/03 v2 layout)
# ===========================================================================
CTD_ROOT     = Path(r"C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\ctd")
CRUISES_ROOT = CTD_ROOT / "cruises"

CRUISE_ID  = "P45_06"
CRUISE_DIR = CRUISES_ROOT / CRUISE_ID

L2_1M_DOWN   = CRUISE_DIR / "L2" / "CTD" / "CTD_1m_down"   # CTD downcast profiles (.cnv)
L2_SUNA_ROOT = CRUISE_DIR / "L2" / "SUNA"                  # merged CTD+SUNA (nitrate)
PLOT_ROOT    = CRUISE_DIR / "L2" / "CTD" / "profile_plots" # figures + lab CSVs out
PLOT_ROOT.mkdir(parents=True, exist_ok=True)

# ===========================================================================
# What to plot. Each entry: (column-name candidates, readable label, unit).
# The plotter uses the first candidate present in the file.
# ===========================================================================
CTD_VARIABLES = [
    (["tv290C", "t090C", "t090Cm", "potemp090C"], "Temperature", "\u00b0C"),
    (["sal00", "sal11"],                          "Salinity",    "PSU"),
    (["sbox0Mm/Kg", "sbeox0Mm/Kg", "sbeox0Mm/L"], "Oxygen",      "\u00b5mol/kg"),
    (["flECO-AFL", "flECO-AFL1"],                 "Fluorescence","mg/m\u00b3"),
    (["CStarAt0"],                                "Beam attenuation", "1/m"),
    (["sigma-\u00e900", "sigma-t00", "density00"],"Density (sigma-t)", "kg/m\u00b3"),
]
NITRATE_VARIABLE = (["suna_nitrate"], "Nitrate", "\u00b5M")

# Depth column candidates (prefer salt-water depth, then pressure).
DEPTH_CANDIDATES = ["depSM", "depSW", "prdM", "prDM", "prM", "pressure"]

# Aesthetics
FIG_DPI = 200
LINE_W  = 1.4
GRID_A  = 0.3
SAVE_FORMATS = ["png"]   # add "pdf" for vector output

# Lab-friendly rename map for the exported CSV (readable columns for colleagues).
LAB_RENAME = {
    "depSM": "Depth_m", "prdM": "Pressure_db", "tv290C": "Temperature_C",
    "sal00": "Salinity_PSU", "sbox0Mm/Kg": "Oxygen_umol_kg",
    "sbeox0Mm/L": "Oxygen_umol_L", "flECO-AFL": "Fluorescence_mg_m3",
    "CStarAt0": "BeamAttenuation_1_m", "par/sat/log": "PAR",
    "potemp090C": "PotentialTemp_C", "sigma-\u00e900": "SigmaT_kg_m3",
    "utc_time": "UTC_time", "suna_nitrate": "Nitrate_uM",
}

print("Cruise      :", CRUISE_ID)
print("CTD 1m down :", L2_1M_DOWN, "(exists:", L2_1M_DOWN.exists(), ")")
print("Merged SUNA :", L2_SUNA_ROOT, "(exists:", L2_SUNA_ROOT.exists(), ")")
print("Plots out   :", PLOT_ROOT)


Cruise      : P45_06
CTD 1m down : C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\ctd\cruises\P45_06\L2\CTD\CTD_1m_down (exists: True )
Merged SUNA : C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\ctd\cruises\P45_06\L2\SUNA (exists: True )
Plots out   : C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\ctd\cruises\P45_06\L2\CTD\profile_plots


## 2. Readers

A minimal `.cnv` parser (same header logic as step02/03) for the CTD downcast profiles, and a helper
to pick the depth column and available variables from any profile dataframe.

In [3]:
def parse_cnv(path: Path) -> pd.DataFrame:
    """Parse a Sea-Bird .cnv into a numeric DataFrame using the header column names."""
    text = path.read_text(encoding="latin-1", errors="replace").splitlines()
    names: Dict[int, str] = {}
    data_start = None
    name_re = re.compile(r"#\s*name\s+(\d+)\s*=\s*([^:]+?)\s*[:=]", re.IGNORECASE)
    for idx, line in enumerate(text):
        s = line.strip()
        if s.startswith("#") or s.startswith("*"):
            m = name_re.search(line)
            if m:
                names[int(m.group(1))] = m.group(2).strip()
            if s.upper().startswith("*END*"):
                data_start = idx + 1
                break
    if data_start is None:
        raise ValueError(f"No *END* in {path}")
    cols = [names[k] for k in sorted(names)]
    rows = []
    for line in text[data_start:]:
        s = line.strip()
        if not s:
            continue
        parts = s.split()
        if len(parts) >= len(cols):
            rows.append(parts[:len(cols)])
    return pd.DataFrame(rows, columns=cols).apply(pd.to_numeric, errors="coerce")


def pick_depth(df: pd.DataFrame) -> Optional[str]:
    for c in DEPTH_CANDIDATES:
        if c in df.columns:
            return c
    # loose fallback
    return next((c for c in df.columns if re.search(r"dep|pr[dD]?M|pressure", c, re.I)), None)


def pick_var(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    return None


## 3. Plot helpers

`plot_single` draws one variable vs depth (depth increasing downward). `plot_overlay` overlays one
variable across all casts. Both save to `PLOT_ROOT` in the configured formats.

In [4]:
def _finish_axis(ax, xlabel, ylabel="Depth (m)"):
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=GRID_A)
    ax.xaxis.set_label_position("top")
    ax.xaxis.tick_top()


def _save(fig, base: Path) -> List[Path]:
    out = []
    for fmt in SAVE_FORMATS:
        p = base.with_suffix(f".{fmt}")
        fig.savefig(p, dpi=FIG_DPI, bbox_inches="tight")
        out.append(p)
    plt.close(fig)
    return out


def plot_single(df: pd.DataFrame, xcol: str, depth_col: str, label: str, unit: str,
                cast_id: str) -> List[Path]:
    d = df[[xcol, depth_col]].apply(pd.to_numeric, errors="coerce").dropna().sort_values(depth_col)
    if d.empty:
        return []
    fig, ax = plt.subplots(figsize=(3.2, 5.0))
    ax.plot(d[xcol], d[depth_col], "-", lw=LINE_W)
    _finish_axis(ax, f"{label} ({unit})")
    ax.set_title(cast_id, fontsize=9)
    slug = re.sub(r"[^A-Za-z0-9]+", "_", label).strip("_").lower()
    return _save(fig, PLOT_ROOT / f"{cast_id}__{slug}")


def plot_overlay(frames: Dict[str, pd.DataFrame], candidates: List[str],
                 label: str, unit: str) -> List[Path]:
    fig, ax = plt.subplots(figsize=(4.0, 6.0))
    n = 0
    for cast_id, df in sorted(frames.items()):
        xcol = pick_var(df, candidates)
        depth_col = pick_depth(df)
        if xcol is None or depth_col is None:
            continue
        d = df[[xcol, depth_col]].apply(pd.to_numeric, errors="coerce").dropna().sort_values(depth_col)
        if d.empty:
            continue
        ax.plot(d[xcol], d[depth_col], "-", lw=1.0, label=cast_id)
        n += 1
    if n == 0:
        plt.close(fig)
        return []
    _finish_axis(ax, f"{label} ({unit})")
    ax.set_title(f"{CRUISE_ID} \u2014 {label} (all casts)", fontsize=10)
    ax.legend(fontsize=6, loc="best")
    slug = re.sub(r"[^A-Za-z0-9]+", "_", label).strip("_").lower()
    return _save(fig, PLOT_ROOT / f"_overlay__{slug}")


## 4. Load all casts

Reads every 1 m downcast CTD profile. For casts that also have a merged SUNA product, attaches the
`suna_nitrate` column (binned to the same depth grid) so nitrate can be plotted alongside CTD.

In [5]:
def attach_nitrate(ctd_df: pd.DataFrame, cast_id: str, depth_col: str) -> pd.DataFrame:
    """If a merged SUNA product exists for this cast, bin its nitrate onto the CTD depth grid."""
    merged_path = L2_SUNA_ROOT / f"{cast_id}_SUNA_1s.csv"
    if not merged_path.exists():
        return ctd_df
    m = pd.read_csv(merged_path)
    mdepth = pick_depth(m)
    if mdepth is None or "suna_nitrate" not in m.columns:
        return ctd_df
    m = m[[mdepth, "suna_nitrate"]].apply(pd.to_numeric, errors="coerce").dropna()
    if m.empty:
        return ctd_df
    # bin SUNA nitrate to 1 m and map onto the CTD profile depths
    m["_bin"] = m[mdepth].round().astype(int)
    binned = m.groupby("_bin")["suna_nitrate"].mean()
    out = ctd_df.copy()
    out["suna_nitrate"] = out[depth_col].round().astype("Int64").map(binned)
    return out


frames: Dict[str, pd.DataFrame] = {}
cast_files = sorted(L2_1M_DOWN.glob("*_1m_down.cnv"))
print(f"Found {len(cast_files)} CTD downcast profiles.")

for f in cast_files:
    cast_id = f.name[: -len("_1m_down.cnv")]
    try:
        df = parse_cnv(f)
        depth_col = pick_depth(df)
        if depth_col is None:
            print(f"  {cast_id}: no depth column, skipped")
            continue
        df = attach_nitrate(df, cast_id, depth_col)
        frames[cast_id] = df
        has_no3 = "suna_nitrate" in df.columns and df["suna_nitrate"].notna().any()
        print(f"  {cast_id}: {len(df)} bins, depth={depth_col}, nitrate={'yes' if has_no3 else 'no'}")
    except Exception as exc:
        print(f"  {cast_id}: FAILED -> {exc!r}")


Found 11 CTD downcast profiles.
  P45_06_CTD_01: 40 bins, depth=depSM, nitrate=no
  P45_06_CTD_02: 165 bins, depth=depSM, nitrate=no
  P45_06_CTD_03: 39 bins, depth=depSM, nitrate=yes
  P45_06_CTD_04: 91 bins, depth=depSM, nitrate=yes
  P45_06_CTD_05: 72 bins, depth=depSM, nitrate=yes
  P45_06_CTD_06: 166 bins, depth=depSM, nitrate=yes
  P45_06_CTD_07: 46 bins, depth=depSM, nitrate=yes
  P45_06_CTD_08: 57 bins, depth=depSM, nitrate=yes
  P45_06_CTD_09: 38 bins, depth=depSM, nitrate=no
  P45_06_CTD_10: 72 bins, depth=depSM, nitrate=no
  P45_06_CTD_11: 197 bins, depth=depSM, nitrate=no


## 5. Per-cast profile plots + lab-friendly CSV

In [6]:
def lab_friendly(df: pd.DataFrame) -> pd.DataFrame:
    keep = [c for c in df.columns if c in LAB_RENAME]
    out = df[keep].rename(columns=LAB_RENAME)
    return out


made: List[Path] = []
for cast_id, df in frames.items():
    depth_col = pick_depth(df)
    # CTD variables
    for candidates, label, unit in CTD_VARIABLES:
        xcol = pick_var(df, candidates)
        if xcol is not None:
            made += plot_single(df, xcol, depth_col, label, unit, cast_id)
    # nitrate where present
    ncol = pick_var(df, NITRATE_VARIABLE[0])
    if ncol is not None and df[ncol].notna().any():
        made += plot_single(df, ncol, depth_col, NITRATE_VARIABLE[1], NITRATE_VARIABLE[2], cast_id)
    # lab-friendly CSV
    lab = lab_friendly(df)
    if not lab.empty:
        lab_path = PLOT_ROOT / f"{cast_id}_readable.csv"
        lab.to_csv(lab_path, index=False, encoding="utf-8-sig")

print(f"Wrote {len(made)} per-cast figure files to {PLOT_ROOT}")


Wrote 72 per-cast figure files to C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\ctd\cruises\P45_06\L2\CTD\profile_plots


## 6. Multi-cast overlays (one figure per variable)

In [7]:
overlay_made: List[Path] = []
for candidates, label, unit in CTD_VARIABLES + [NITRATE_VARIABLE]:
    overlay_made += plot_overlay(frames, candidates, label, unit)

print(f"Wrote {len(overlay_made)} overlay figures.")
print("\nAll outputs in:", PLOT_ROOT)
for p in sorted(PLOT_ROOT.glob("_overlay__*")):
    print("  ", p.name)


Wrote 7 overlay figures.

All outputs in: C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\ctd\cruises\P45_06\L2\CTD\profile_plots
   _overlay__beam_attenuation.png
   _overlay__density_sigma_t.png
   _overlay__fluorescence.png
   _overlay__nitrate.png
   _overlay__oxygen.png
   _overlay__salinity.png
   _overlay__temperature.png
